<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Duckiedrone Process Inspection

Finding a program’s files on the Duckiedrone does not tell you whether that program is running. Likewise, a slow dashboard or camera stream needs more investigation than checking that its directory exists.

You have already practiced inspecting processes in your local Linux environment. In this notebook, you will apply those tools to a physical or virtual Duckiedrone: read a process listing, distinguish processes from service names, and observe resource consumption (very important for debugging!)without changing running programs.

## Inspect Duckiedrone services

Open a (physical or virtual) Duckiedrone shell using the connection method from the previous notebooks, then confirm your context:

```bash
hostname
whoami
```

Process visibility depends on where the shell runs. [Figure 1](#figure-1) shows the contexts you may encounter.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
Base-station shell
        |
        | SSH or virtual connect
        v
Duckiedrone shell
        |
        | Service-specific tools
        v
Service shell or workbench
  </pre>
  <figcaption>Figure 1: Shell contexts used when inspecting a robot.</figcaption>
</figure>

Run `dts` commands from the base station (the robots do not have `dts` installed). For this activity, stay in the Duckiedrone shell. 

### Read a process snapshot

Run:

```bash
ps -ef
```

`-e` selects all processes visible in this environment; `-f` requests the full listing format.

An illustrative excerpt from an SSH session is:

```text
UID        PID  PPID  C STIME TTY          TIME CMD
duckie    2150  2149  0 10:20 pts/0    00:00:00 -bash
duckie    2192  2150  0 10:21 pts/0    00:00:00 ps -ef
```

Focus on four columns:

| Column | Meaning in this example |
| --- | --- |
| `UID` | Both processes run under the `duckie` account |
| `PID` | The shell is process `2150`; `ps` is process `2192` |
| `PPID` | The parent of `ps` is the shell, process `2150` |
| `CMD` | The command associated with each process |

`TIME` is accumulated CPU time, not elapsed time since startup. A mostly idle shell can have been open for several minutes while displaying little CPU time.

The listing includes `ps` itself because it was running when the snapshot was taken. Your identifiers and times will differ. To isolate your current Bash shell, run:

```bash
ps -f -p "$$"
```

`$$` expands to the shell’s own PID, and `-p` selects that process. See the [`ps` manual](https://man7.org/linux/man-pages/man1/ps.1.html) for the listing fields and options.

### Connect processes to robot functions

Duckiedrone software performs several jobs:

| Function | Example |
| --- | --- |
| Platform management | Store settings and report device status |
| Device access | Obtain camera images or range measurements |
| Communication | Exchange data between drivers and other components |
| User interface | Serve the dashboard |

You may encounter service names such as `driver-camera`, `driver-tof-bottom`, `dtps`, and `dashboard`. These are not guaranteed to appear literally in `ps`: a service might run several processes, and its executable might have a generic name such as `python3`.

For example, finding a Python process does not establish that the camera service is healthy. Conversely, failing to find the text `driver-camera` does not establish that the service is absent.

## Monitor processes read-only

`ps` gives a snapshot. To see whether resource use changes over time, check for an interactive monitor:

```bash
command -v htop
```

If this prints a path, such as `/usr/bin/htop`, run:

```bash
htop
```

This will open a dynamic visualization of the processes running, the resources they consume, and the overall status of the device. Observe the display for about 15 seconds, then press **q** or **F10** to leave. See the [`htop` manual](https://man7.org/linux/man-pages/man1/htop.1.html) for more detailed features.

If `htop` is unavailable or does not support that option, use the Ubuntu `top` fallback:

```bash
top -s
```

`-s` enables secure mode, disabling actions such as killing processes and changing their priority. Press **q** to leave. The [`top` manual](https://man7.org/linux/man-pages/man1/top.1.html) describes this mode.

In either monitor, focus on one process and watch its CPU and memory readings across several updates.

For example, a brief CPU increase while a service starts is different from sustained high use while a camera stream remains unresponsive. Record what you observed and what the robot was doing; a high reading alone does not identify a fault.

### Try it

Collect one short observation:

1. Identify your current shell in the process listing.
2. Observe one process in the monitor over several updates.
3. Record its name, whether CPU use was steady or changing, and whether you observed a related symptom.

An illustrative note is:

> From the physical Duckiedrone shell, a Python process showed changing CPU use over 15 seconds. The dashboard remained responsive. I have not identified which service owns that process.

This separates observation from interpretation.

<details>
<summary>Check your result</summary>

You should be able to identify the environment you inspected, interpret a process row, and describe a resource-use observation. A process being present does not prove its service works correctly, and an absent name does not prove a service has stopped.

No process needs to be killed, restarted, or reconfigured to complete this activity.

</details>

## Further reading

The Linux manuals for [`ps`](https://man7.org/linux/man-pages/man1/ps.1.html), [`htop`](https://man7.org/linux/man-pages/man1/htop.1.html), and [`top`](https://man7.org/linux/man-pages/man1/top.1.html) explain process fields and monitoring controls.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
